In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "SMQFAM_E.xpt"
csv_file = DATA_DIR / "SMQFAM_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

source_df = df.copy()


print("=" * 75)
print("NHANES 2007-2008 SMQFAM_E XPT -> CSV")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SMD410",
    "SMD415",
    "SMD415A",
    "SMD430"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "SMQFAM_E column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 10149:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 5:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 10,149 rows x 5 columns."
)


# ============================================================
# 4. CHECK FOR KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)

true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)

print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print("\nTiny values by variable:")

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Tiny-value artifacts were unexpectedly "
        "found in SMQFAM_E. Do not automatically replace."
    )


print(
    "PASS: No tiny XPORT zero-decoding artifact exists."
)


# ============================================================
# 5. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# IMPORTANT:
#
# Never round real measurements or legitimate decimal values.
#
# This check is performed BEFORE any conversion to Int64.
# If genuine decimal values exist, the program stops instead
# of rounding or changing them.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = df[col].dropna()

    if values.empty:
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"\nGenuine decimal values in {col}:"
        )

        print(
            values[
                fractional_mask
            ].head(20)
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values detected. "
        "They will NOT be rounded or changed:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values "
    "exist in SMQFAM_E."
)


# ============================================================
# 6. CLEAN SEQN
# ============================================================

seqn = df["SEQN"].dropna()


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(df["SEQN"].isna().sum()) != 0
    or
    int(df["SEQN"].duplicated().sum()) != 0
    or
    df["SEQN"].min() != 41475
    or
    df["SEQN"].max() != 51623
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 7. FORMAT QUESTIONNAIRE VALUES WITHOUT .0
# ============================================================

#
# All four SMQFAM_E questionnaire variables contain
# whole-number codes/counts in this released file.
#
# The fractional-value check above proves that conversion
# to Int64 is safe for THIS file.
#
# No .round() is used.
#

DATA_COLUMNS = [
    col
    for col in df.columns
    if col != "SEQN"
]


for col in DATA_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: Questionnaire/count fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 8. CDC CHECK - SMD410
# ============================================================

#
# CDC:
#
# 1 = Yes       : 1,840
# 2 = No        : 8,230
# 7 = Refused   : 2
# 9 = Don't know: 0
# Missing       : 77
#

EXPECTED_SMD410 = {
    1: 1840,
    2: 8230,
    7: 2
}


actual = (
    df["SMD410"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- SMD410 ---")


if (
    actual != EXPECTED_SMD410
    or
    int(df["SMD410"].eq(9).sum()) != 0
    or
    int(df["SMD410"].isna().sum()) != 77
):

    raise ValueError(
        "SMD410 differs from CDC."
    )


print(
    "PASS: SMD410 matches CDC."
)


# ============================================================
# 9. CDC CHECK - SMD415
# ============================================================

#
# CDC:
#
# 1-2 = 1,635
# 3   = 205
# Missing = 8,309
#

s = df["SMD415"]


print("\n--- SMD415 ---")

print(
    "1-2:",
    int(s.between(1, 2).sum())
)

print(
    "3 or more:",
    int(s.eq(3).sum())
)

print(
    "Missing:",
    int(s.isna().sum())
)


if not (
    int(s.between(1, 2).sum()) == 1635
    and
    int(s.eq(3).sum()) == 205
    and
    int(s.isna().sum()) == 8309
):

    raise ValueError(
        "SMD415 differs from CDC."
    )


print(
    "PASS: SMD415 matches CDC."
)


# ============================================================
# 10. CDC CHECK - SMD415A
# ============================================================

#
# CDC:
#
# 1-2 = 1,546
# 3   = 191
# Missing = 8,412
#

s = df["SMD415A"]


print("\n--- SMD415A ---")

print(
    "1-2:",
    int(s.between(1, 2).sum())
)

print(
    "3 or more:",
    int(s.eq(3).sum())
)

print(
    "Missing:",
    int(s.isna().sum())
)


if not (
    int(s.between(1, 2).sum()) == 1546
    and
    int(s.eq(3).sum()) == 191
    and
    int(s.isna().sum()) == 8412
):

    raise ValueError(
        "SMD415A differs from CDC."
    )


print(
    "PASS: SMD415A matches CDC."
)


# ============================================================
# 11. CDC CHECK - SMD430
# ============================================================

#
# CDC released frequencies:
#
# 1-38 = 1,528
# 40   = 209
# Missing = 8,412
#
# Questionnaire instructions say:
# "IF NONE, ENTER 0"
#
# However, the released 2007-2008 file contains
# NO zero records for SMD430.
#

s = df["SMD430"]


print("\n--- SMD430 ---")

print(
    "1-38:",
    int(s.between(1, 38).sum())
)

print(
    "40 or more:",
    int(s.eq(40).sum())
)

print(
    "Zero:",
    int(s.eq(0).sum())
)

print(
    "Missing:",
    int(s.isna().sum())
)


if not (
    int(s.between(1, 38).sum()) == 1528
    and
    int(s.eq(40).sum()) == 209
    and
    int(s.eq(0).sum()) == 0
    and
    int(s.isna().sum()) == 8412
):

    raise ValueError(
        "SMD430 differs from CDC."
    )


print(
    "PASS: SMD430 matches CDC."
)


# ============================================================
# 12. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# IMPORTANT:
#
# There is NO zero restoration in this file.
#
# Expected values are therefore exactly the original XPT.
# Int64 conversion only changes display/CSV representation,
# not the numeric values.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in source_df.columns:

    source_values = (
        source_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        source_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes in:\n"
        + str(failures)
    )


print(
    "PASS: All 5 variables preserve "
    "the original XPT values exactly."
)


# ============================================================
# 13. EXPORT CSV
# ============================================================

#
# No float_format.
# No rounding.
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 14. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 15. EXACT XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 5 columns match source XPT values."
)


# ============================================================
# 16. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 17. RAW CSV FORMATTING
# ============================================================

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 18. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("SMQFAM_E CONVERSION COMPLETE")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("Tiny-value corrections: NONE")
print("Genuine decimal variables: NONE")
print("XPT -> CSV differences: 0 expected")
print("Missing-value differences: 0 expected")
print("Unnecessary .0: 0 expected")

NHANES 2007-2008 SMQFAM_E XPT -> CSV
Rows: 10,149
Columns: 5
PASS: Structure = 10,149 rows x 5 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 0
PASS: No tiny XPORT zero-decoding artifact exists.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in SMQFAM_E.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 41475
Maximum: 51623
PASS: SEQN validated.

PASS: Questionnaire/count fields formatted without unnecessary .0.

--- SMD410 ---
PASS: SMD410 matches CDC.

--- SMD415 ---
1-2: 1635
3 or more: 205
Missing: 8309
PASS: SMD415 matches CDC.

--- SMD415A ---
1-2: 1546
3 or more: 191
Missing: 8412
PASS: SMD415A matches CDC.

--- SMD430 ---
1-38: 1528
40 or more: 209
Zero: 0
Missing: 8412
PASS: SMD430 matches CDC.

--- VALUE FIDELITY BEFORE EXPORT ---
PASS: All 5 variables preserve the original XPT values exactly.

CSV created:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_ML_Data_Predictor\Clinical_Neighbourhood_ML_Data_